In this code, I will look to merge all of the data files so that we have one dataset in which each row corresponds to a single individual, their demographic information, and their dietary information. This construction is performed in three parts. The first part is organizing the dietary information. The second part is organizing the demographic information. The third part is merging these two components. 

Additionally, I will construct a food file, which will take the form of a single food item and its associated nutritional values. This will be important for the final application and possibly stretch extensions if we have enough time.

# Reading in the Data

We begin by reading in all of our data files:

In [176]:
import pandas as pd
from pathlib import Path
import numpy as np

In [177]:
raw = Path("data/raw/2017_2018")

# Read function
def read(path):
    # .xpt
    if path.suffix.lower() == ".xpt":
        df = pd.read_sas(path, format="xport", encoding="utf-8")
    
    # sas
    elif path.suffix.lower() == ".sas7bdat":
        df = pd.read_sas(path, encoding="utf-8")
    
    # excel
    else:
        df = pd.read_excel(path)
    
    # Normalize to capitals
    df.columns = df.columns.str.upper()

    # Cast SEQN to integer (from float)
    if "SEQN" in df.columns:
        df["SEQN"] = df["SEQN"].astype(int)

    # Strange 0 reading happens with xpt where
    # 0s are not 0s but very tiny numbers
    num = df.select_dtypes("number").columns
    df[num] = df[num].mask(df[num].abs() < 1e-30, 0)

    return df

# NHANES files
nhanes_names = ["DEMO", "DR1IFF", "DR2IFF", "DR1TOT", "DRXFCD", "GHB", "HDL", "TCHOL",
                "HSCRP", "BMX", "BPX", "DIQ", "SMQ", "PAQ", "FSQ"]

# NHANES dictionary, where file name maps to dataframe
# Each glob will only return one object, so we just take the one object

data = {}

for name in nhanes_names:
    print(name)
    data[name] = read(next(raw.glob(f"{name}_J.*")))

# USDA files
usda_files = {
    "FPED_DB":      "FPED_1718.xls",
    "FLAV_VAL":     "flavval.sas7bdat",
    "FLAV_DESC":    "flavdesc.sas7bdat",
    "FLAV_FOODS":   "mainfooddesc.sas7bdat",
    "FPED_TOT1":    "fped_dr1tot_1718.sas7bdat",
    "FPED_TOT2":    "fped_dr2tot_1718.sas7bdat",
    "FLAV_TOT1":    "flav_dr1tot_1718.sas7bdat",
    "FLAV_TOT2":    "flav_dr2tot_1718.sas7bdat",
}

# More of the same, it only looks different because we are adding to a pre-existing dictionary
for name, file in usda_files.items():
    data[name] = read(raw / file)

# Summarize dataframes
for name, df in data.items():
    people = df["SEQN"].nunique() if "SEQN" in df.columns else "-"
    print(f"{name:11} rows={len(df):>8,}  people={people:>6}  cols={df.shape[1]}")

DEMO
DR1IFF
DR2IFF
DR1TOT
DRXFCD
GHB
HDL
TCHOL
HSCRP
BMX
BPX
DIQ
SMQ
PAQ
FSQ
WARNING *** OLE2 inconsistency: SSCS size is 0 but SSAT size is non-zero
DEMO        rows=   9,254  people=  9254  cols=46
DR1IFF      rows= 112,683  people=  7640  cols=84
DR2IFF      rows=  93,500  people=  6639  cols=84
DR1TOT      rows=   8,704  people=  8704  cols=168
DRXFCD      rows=   7,083  people=     -  cols=3
GHB         rows=   6,401  people=  6401  cols=2
HDL         rows=   7,435  people=  7435  cols=3
TCHOL       rows=   7,435  people=  7435  cols=3
HSCRP       rows=   8,366  people=  8366  cols=3
BMX         rows=   8,704  people=  8704  cols=21
BPX         rows=   8,704  people=  8704  cols=21
DIQ         rows=   8,897  people=  8897  cols=54
SMQ         rows=   6,724  people=  6724  cols=37
PAQ         rows=   5,856  people=  5856  cols=17
FSQ         rows=   9,254  people=  9254  cols=49
FPED_DB     rows=   7,083  people=     -  cols=39
FLAV_VAL    rows= 262,071  people=     -  cols=5
FLAV_

# Constructing Diets

In [178]:
# Inspect USDA lookup tables and IFF structure
for name in ["FPED_DB", "FLAV_VAL", "FLAV_DESC", "FLAV_FOODS"]:
    df = data[name]
    print(f"\n=== {name}  {df.shape} ===")
    print(df.dtypes.to_string())
    print(df.head(3).to_string())

# First 20 columns of each IFF file, to check prefixes before stacking
for name in ["DR1IFF", "DR2IFF"]:
    print(f"\n=== {name} first 20 columns ===")
    print(data[name].columns[:20].tolist())


=== FPED_DB  (7083, 39) ===
FOODCODE                        int64
DESCRIPTION                       str
F_TOTAL (CUP EQ.)             float64
F_CITMLB (CUP EQ.)            float64
F_OTHER (CUP EQ.)             float64
F_JUICE (CUP EQ.)             float64
V_TOTAL (CUP EQ.)             float64
V_DRKGR (CUP EQ.)             float64
V_REDOR_TOTAL (CUP EQ.)       float64
V_REDOR_TOMATO (CUP EQ.)      float64
V_REDOR_OTHER (CUP EQ.)       float64
V_STARCHY_TOTAL (CUP EQ.)     float64
V_STARCHY_POTATO (CUP EQ.)    float64
V_STARCHY_OTHER (CUP EQ.)     float64
V_OTHER (CUP EQ.)             float64
V_LEGUMES (CUP EQ.)           float64
G_TOTAL (OZ. EQ.)             float64
G_WHOLE (OZ. EQ.)             float64
G_REFINED (OZ. EQ.)           float64
PF_TOTAL (OZ. EQ.)            float64
PF_MPS_TOTAL (OZ. EQ.)        float64
PF_MEAT (OZ. EQ.)             float64
PF_CUREDMEAT (OZ. EQ.)        float64
PF_ORGAN (OZ. EQ.)            float64
PF_POULT (OZ. EQ.)            float64
PF_SEAFD_HI (OZ. EQ.)

We will look to prune the FPED DB to make merging tidier:

In [179]:
data["FPED_DB"]["DESCRIPTION"]

0                                    Milk, human
1                                      Milk, NFS
2                                    Milk, whole
3                        Milk, low sodium, whole
4                 Milk, calcium fortified, whole
                          ...                   
7078            Tomatoes as ingredient in omelet
7079    Other vegetables as ingredient in omelet
7080           Vegetables as ingredient in curry
7081           Sauce as ingredient in hamburgers
7082        Industrial oil as ingredient in food
Name: DESCRIPTION, Length: 7083, dtype: str

In [180]:
# FPED lookup table, one row per food code, values per 100 g
# We drop description column and extract units for future use
# We drop units from name for concision
fped = data["FPED_DB"].drop(columns="DESCRIPTION").copy()
raw_cols = fped.columns
fped.columns = raw_cols.str.replace(r"\s*\(.*\)$", "", regex=True)

# fped_units["F_TOTAL"] -> "CUP EQ."
# possible later use
fped_units = dict(zip(fped.columns, raw_cols.str.extract(r"\((.*)\)$")[0]))

# store fped columns
fped_cols = [c for c in fped.columns if c != "FOODCODE"]

# Check that we just have unique food codes (validation step)
assert fped["FOODCODE"].is_unique
print(fped.shape, fped_cols[:5])

(7083, 38) ['F_TOTAL', 'F_CITMLB', 'F_OTHER', 'F_JUICE', 'V_TOTAL']


Now, we clean up the flavanoid table:

In [181]:
data["FLAV_VAL"].head()

,FOOD_CODE,NUTRIENT_CODE,START_DATE,END_DATE,NUTRIENT_VALUE
0,11000000.0,710.0,2017-01-01,2018-12-31,0.0
1,11000000.0,711.0,2017-01-01,2018-12-31,0.0
2,11000000.0,712.0,2017-01-01,2018-12-31,0.0
3,11000000.0,731.0,2017-01-01,2018-12-31,0.0
4,11000000.0,740.0,2017-01-01,2018-12-31,0.0


In [182]:
# AI-generated (Claude): flavonoid lookup table, one row per food code, mg per 100 g
flv = data["FLAV_VAL"].copy()

# One release period across the entire table, validation step
assert flv["START_DATE"].nunique() == 1 and flv["END_DATE"].nunique() == 1 

# Convert foodcode to integer, nutrient code to integer
flv["FOODCODE"] = flv["FOOD_CODE"].astype(int)
flv["NUTRIENT_CODE"] = flv["NUTRIENT_CODE"].astype(int)

# Creates a dataframe table that maps nutrient code to flav short name
tags = (data["FLAV_DESC"]
        .assign(NUTRIENT_CODE=lambda d: d["NUTRIENT_CODE"].astype(int))
        .set_index("NUTRIENT_CODE")["TAGNAME"])

# As seen above, food codes are scattered across multiple rows
# We can fix this by performing a pivot to collapse a food into a single row
# pivot raises an error if any food/flavonoid pair is duplicated, which doubles as a check
flav = flv.pivot(index="FOODCODE", columns="NUTRIENT_CODE", values="NUTRIENT_VALUE")
flav.columns = "FL_" + flav.columns.map(tags)
flav = flav.reset_index()
flav_cols = [c for c in flav.columns if c != "FOODCODE"]

assert flav[flav_cols].notna().all().all()
print(flav.shape)  # expect (7083, 38)

(7083, 38)


Strange missing values as column headers:

In [183]:
flav.info()

<class 'pandas.DataFrame'>
RangeIndex: 7083 entries, 0 to 7082
Data columns (total 38 columns):
 #   Column         Non-Null Count  Dtype  
---  ------         --------------  -----  
 0   FOODCODE       7083 non-null   int64  
 1   FL_DAIDZN      7083 non-null   float64
 2   FL_GNSTEIN     7083 non-null   float64
 3   FL_GLYCTEIN    7083 non-null   float64
 4   FL_CYAD        7083 non-null   float64
 5   FL_PETUNIDIN   7083 non-null   float64
 6   FL_DELPH       7083 non-null   float64
 7   FL_MALVIDIN    7083 non-null   float64
 8   nan            7083 non-null   float64
 9   FL_PEONIDIN    7083 non-null   float64
 10  nan            7083 non-null   float64
 11  FL_EPICATEGC   7083 non-null   float64
 12  FL_EPICATEC    7083 non-null   float64
 13  FL_EPICATECG3  7083 non-null   float64
 14  nan            7083 non-null   float64
 15  nan            7083 non-null   float64
 16  nan            7083 non-null   float64
 17  nan            7083 non-null   float64
 18  FL_HESPT       7083

Let's see what's going on:

In [184]:
# AI-generated (Claude): diagnose unnamed flavonoid columns
# Make a new desc copy to compare original names to remaining
desc = data["FLAV_DESC"].copy()
desc["NUTRIENT_CODE"] = desc["NUTRIENT_CODE"].astype(int)

# val codes - what remains in our flv
val_codes = set(flv["NUTRIENT_CODE"].unique())

# desc codes - what originally exists in our desc
desc_codes = set(desc["NUTRIENT_CODE"])

print("In values but not in FLAV_DESC:", sorted(val_codes - desc_codes))
print("In FLAV_DESC but not in values:", sorted(desc_codes - val_codes))
print("\nRows with blank TAGNAME:")
print(desc[desc["TAGNAME"].isna()].to_string())
print("\nFull FLAV_DESC:")
print(desc[["NUTRIENT_CODE", "FLAVONOID_DESCRIPTION", "TAGNAME"]].to_string())

In values but not in FLAV_DESC: []
In FLAV_DESC but not in values: []

Rows with blank TAGNAME:
    NUTRIENT_CODE                          FLAVONOID_DESCRIPTION FLAVONOID_CLASS TAGNAME UNIT  DECIMALS
7             743                                   Pelargonidin  Anthocyanidins     NaN   mg       2.0
9             749                                   (+)-Catechin    Flavan-3-ols     NaN   mg       2.0
13            753                 (-)-Epigallocatechin 3-gallate    Flavan-3-ols     NaN   mg       2.0
14            755                                     Theaflavin    Flavan-3-ols     NaN   mg       2.0
15            756                                   Thearubigins    Flavan-3-ols     NaN   mg       2.0
16            758                                    Eriodictyol      Flavanones     NaN   mg       2.0
21            785                                   Isorhamnetin       Flavonols     NaN   mg       2.0
25            791                      Theaflavin-3,3'-digallate    Flav

So, none were lost, they just didn't have a tagname. I'll use Claude to replace them with sensible tag names:

In [185]:
# AI-generated (Claude): flavonoid lookup table, one row per food code, mg per 100 g
flv = data["FLAV_VAL"].copy()

# One release period across the entire table, validation step
assert flv["START_DATE"].nunique() == 1 and flv["END_DATE"].nunique() == 1

# Convert foodcode to integer, nutrient code to integer
flv = flv.rename(columns={"FOOD_CODE": "FOODCODE"})
flv["FOODCODE"] = flv["FOODCODE"].astype(int)
flv["NUTRIENT_CODE"] = flv["NUTRIENT_CODE"].astype(int)

# Creates a table that maps nutrient code to flav short name
# USDA left TAGNAME blank for 18 codes, so we supply names for those
desc = data["FLAV_DESC"].copy()                              
desc["NUTRIENT_CODE"] = desc["NUTRIENT_CODE"].astype(int)

missing_tags = {                                           
    743: "PELARGON",      749: "CATECHIN",      753: "EGCG",
    755: "THEAFLAV",      756: "THEARUBIG",     758: "ERIODICT",
    785: "ISORHAM",       791: "THEAFLAV33DG",  792: "THEAFLAV3PG",
    793: "THEAFLAV3G",
    7000: "TOT_FLAVONOIDS",  7100: "TOT_ANTHOCYAN", 7200: "TOT_CATECHINS",
    7300: "TOT_FLAVAN3OL",   7400: "TOT_FLAVANONE", 7500: "TOT_FLAVONE",
    7600: "TOT_FLAVONOL",    7700: "TOT_ISOFLAV",
}

# NUTRIENT_CODE : TAGNAME, replace missing based on NUTRIENT_CODE
tags = desc.set_index("NUTRIENT_CODE")["TAGNAME"].fillna(pd.Series(missing_tags)) 
assert tags.notna().all() and tags.is_unique  # All tags are unique and non-null                 

# As seen above, food codes are scattered across multiple rows
# We can fix this by performing a pivot to collapse a food into a single row
# pivot raises an error if any food/flavonoid pair is duplicated, which doubles as a check
flav = flv.pivot(index="FOODCODE", columns="NUTRIENT_CODE", values="NUTRIENT_VALUE")
flav.columns = "FL_" + flav.columns.map(tags)
assert flav.columns.notna().all() and flav.columns.is_unique # Columns are unique and all non-null names
flav = flav.reset_index()

# Totals (codes 7000-7700) are sums of the 29 compounds
flav_cols = [c for c in flav.columns if c != "FOODCODE"]
flav_total_cols = [c for c in flav_cols if c.startswith("FL_TOT_")]        # 8 totals
flav_compound_cols = [c for c in flav_cols if not c.startswith("FL_TOT_")] # 29 compounds

# Make sure that all columns are not na
assert flav[flav_cols].notna().all().all()
print(flav.shape)  # expect (7083, 38)

# Check: total flavonoids should equal the sum of the 29 compounds for every food   # NEW
gap = (flav[flav_compound_cols].sum(axis=1) - flav["FL_TOT_FLAVONOIDS"]).abs()
print(f"max gap: {gap.max():.4f}  foods off by >0.01 mg: {(gap > 0.01).sum()}")

(7083, 38)
max gap: 0.0000  foods off by >0.01 mg: 0


In [186]:
# AI-generated (Claude)

def prep_iff(df, day):
    # Make a copy
    out = df.copy()

    # Strip day prefix for columns so that the days share the same column names
    # Ex: DR1IGRMS -> GRMS, DR2IGRMS -> GRMS
    out.columns = out.columns.str.replace(r"^DR[12]I?", "", regex=True)

    # Rename to more readable names
    out = out.rename(columns={
        "DAY":   "DOW",
        "FDCD":  "FOODCODE",
        "GRMS":  "GRAMS",
        "_020":  "TIME_EATEN",
        "_030Z": "OCCASION",
        "FS":    "SOURCE",
        "_040Z": "AT_HOME",
        "CCMNM": "COMBO_ID",
        "CCMTX": "COMBO_TYPE",
    })
    # Add day
    out.insert(1, "RECALL_DAY", day)
    return out

# Apply to DRFF tables
iff1 = prep_iff(data["DR1IFF"], 1)
iff2 = prep_iff(data["DR2IFF"], 2)

# Check columns are ready for merge
assert list(iff1.columns) == list(iff2.columns), "column names differ after stripping"

# Stack tables
foods = pd.concat([iff1, iff2], ignore_index=True)

# Make sure no duplicated columns
assert not foods.columns.duplicated().any()

# Filter out for only reliable recalls
foods = foods[foods["DRSTZ"] == 1].copy()

# Coerce foodcode
foods["FOODCODE"] = foods["FOODCODE"].astype(int)

# Summarize
print(foods.shape)
print("People per recall day:", foods.groupby("RECALL_DAY")["SEQN"].nunique().to_dict())
print(foods[["SEQN", "RECALL_DAY", "FOODCODE", "GRAMS", "OCCASION", "SOURCE", "AT_HOME"]].head())

(202670, 85)
People per recall day: {1: 7483, 2: 6502}
    SEQN  RECALL_DAY  FOODCODE   GRAMS  OCCASION  SOURCE  AT_HOME
0  93704           1  55100050   80.58       1.0     3.0      1.0
1  93704           1  11513600  248.00       1.0     1.0      1.0
2  93704           1  24198739  100.00       2.0     1.0      2.0
3  93704           1  74401010    5.67       2.0     1.0      2.0
4  93704           1  71401020  112.00       2.0     1.0      2.0


In [187]:
foods.info()

<class 'pandas.DataFrame'>
Index: 202670 entries, 0 to 206182
Data columns (total 85 columns):
 #   Column      Non-Null Count   Dtype  
---  ------      --------------   -----  
 0   SEQN        202670 non-null  int64  
 1   RECALL_DAY  202670 non-null  int64  
 2   WTDRD1      202670 non-null  float64
 3   WTDR2D      202670 non-null  float64
 4   LINE        202670 non-null  float64
 5   DRSTZ       202670 non-null  float64
 6   EXMER       202670 non-null  float64
 7   DRABF       202670 non-null  float64
 8   DRDINT      202670 non-null  float64
 9   DBIH        195980 non-null  float64
 10  DOW         202670 non-null  float64
 11  LANG        202670 non-null  float64
 12  COMBO_ID    202670 non-null  float64
 13  COMBO_TYPE  202670 non-null  float64
 14  TIME_EATEN  202670 non-null  float64
 15  OCCASION    202670 non-null  float64
 16  SOURCE      190133 non-null  float64
 17  AT_HOME     202670 non-null  float64
 18  FOODCODE    202670 non-null  int64  
 19  GRAMS       202670

In [188]:
# Perform a join with foods (each row represents a food a certain person ate)
# first with fped
# second with flav
# at both points, we add a missing indicator to ensure that all foods have corresponding fped and flav counts 
foods = foods.merge(fped, on="FOODCODE", how="left", validate="m:1", indicator="IN_FPED")
foods = foods.merge(flav, on="FOODCODE", how="left", validate="m:1", indicator="IN_FLAV")

# check output
print("Missing FPED:", (foods["IN_FPED"] == "left_only").sum(),
      "| Missing flavonoids:", (foods["IN_FLAV"] == "left_only").sum())

# Drop indicators
foods = foods.drop(columns=["IN_FPED", "IN_FLAV"])

# Locate all columns 
attr_cols = fped_cols + flav_cols

# Currently, attr_cols stores the per 100g amount of each nutrient
# foods["Grams"] stores the total grams consumed of the food
# So, we divide foods["Grams"] by 100 to get total food consumed in 100s of grams
# This command will then replace the rates with the flat totals
foods[attr_cols] = foods[attr_cols].mul(foods["GRAMS"] / 100, axis=0)

print(foods.shape)
print(foods[["SEQN", "FOODCODE", "GRAMS", "F_TOTAL", "G_WHOLE", "FL_TOT_FLAVONOIDS"]].head())

Missing FPED: 0 | Missing flavonoids: 0
(202670, 159)
    SEQN  FOODCODE   GRAMS  F_TOTAL  G_WHOLE  FL_TOT_FLAVONOIDS
0  93704  55100050   80.58      0.0      0.0           0.000000
1  93704  11513600  248.00      0.0      0.0           1.909600
2  93704  24198739  100.00      0.0      0.0           0.000000
3  93704  74401010    5.67      0.0      0.0           0.058968
4  93704  71401020  112.00      0.0      0.0           2.587200


Great! No missing information for food codes. Lets perform some validation to make sure that our calorie counts look right. NHANES actually provides an aggregation, but for later reasons, we may not want to use that pre-aggregation. Nonetheless, it will serve as a solid check that our pipeline of aggregation worked:

In [189]:
# AI-generated (Claude)

# For day 1, group by people, count total calories
d1 = foods.loc[foods["RECALL_DAY"] == 1].groupby("SEQN")["KCAL"].sum()

# Extract calories and individual seqn as index
tot = data["DR1TOT"].set_index("SEQN")["DR1TKCAL"]

# Concatenate by index and drop missing
cmp = pd.concat([d1, tot], axis=1, join="inner").dropna()

diff = (cmp["KCAL"] - cmp["DR1TKCAL"]).abs()
print(f"n={len(cmp)}  max abs diff={diff.max():.3f}  people off by >1 kcal: {(diff > 1).sum()}")

n=7483  max abs diff=0.000  people off by >1 kcal: 0


In [190]:
print(data["FPED_TOT1"].columns.tolist())
print(data["FLAV_TOT1"].columns.tolist())

['SEQN', 'RIAGENDR', 'RIDAGEYR', 'RIDRETH3', 'SDMVPSU', 'SDMVSTRA', 'INDFMIN2', 'INDFMPIR', 'WTDRD1', 'WTDR2D', 'DR1DRSTZ', 'DRABF', 'DRDINT', 'DR1TNUMF', 'DR1T_F_TOTAL', 'DR1T_F_CITMLB', 'DR1T_F_OTHER', 'DR1T_F_JUICE', 'DR1T_V_TOTAL', 'DR1T_V_DRKGR', 'DR1T_V_REDOR_TOTAL', 'DR1T_V_REDOR_TOMATO', 'DR1T_V_REDOR_OTHER', 'DR1T_V_STARCHY_TOTAL', 'DR1T_V_STARCHY_POTATO', 'DR1T_V_STARCHY_OTHER', 'DR1T_V_OTHER', 'DR1T_V_LEGUMES', 'DR1T_G_TOTAL', 'DR1T_G_WHOLE', 'DR1T_G_REFINED', 'DR1T_PF_TOTAL', 'DR1T_PF_MPS_TOTAL', 'DR1T_PF_MEAT', 'DR1T_PF_CUREDMEAT', 'DR1T_PF_ORGAN', 'DR1T_PF_POULT', 'DR1T_PF_SEAFD_HI', 'DR1T_PF_SEAFD_LOW', 'DR1T_PF_EGGS', 'DR1T_PF_SOY', 'DR1T_PF_NUTSDS', 'DR1T_PF_LEGUMES', 'DR1T_D_TOTAL', 'DR1T_D_MILK', 'DR1T_D_YOGURT', 'DR1T_D_CHEESE', 'DR1T_OILS', 'DR1T_SOLID_FATS', 'DR1T_ADD_SUGARS', 'DR1T_A_DRINKS']
['SEQN', 'DR1DRSTZ', 'DR1T_FL710', 'DR1T_FL711', 'DR1T_FL712', 'DR1T_FL731', 'DR1T_FL740', 'DR1T_FL741', 'DR1T_FL742', 'DR1T_FL743', 'DR1T_FL745', 'DR1T_FL749', 'DR1T_FL750'

In [191]:
set(d1.index) - set(tot.index)

set()

In [192]:
set(tot.index) - set(d1.index)

{94209,
 102402,
 100355,
 102408,
 94217,
 98314,
 96266,
 94221,
 100366,
 98319,
 102414,
 94224,
 100370,
 100371,
 100372,
 96277,
 100375,
 100380,
 98333,
 98335,
 100384,
 98342,
 94246,
 98344,
 94250,
 98347,
 94254,
 102448,
 98355,
 98356,
 100403,
 98358,
 100407,
 102451,
 96307,
 96314,
 98363,
 98365,
 102462,
 94269,
 96318,
 102465,
 94273,
 96321,
 94276,
 96327,
 96328,
 102474,
 102476,
 100429,
 94284,
 100431,
 98386,
 98387,
 102486,
 96342,
 100440,
 102490,
 94298,
 100444,
 98397,
 98400,
 96353,
 94307,
 100452,
 98406,
 102502,
 94311,
 98409,
 102507,
 94315,
 100465,
 94327,
 98424,
 94328,
 102523,
 94331,
 96381,
 102527,
 100480,
 102529,
 94335,
 94338,
 96386,
 94341,
 102535,
 102536,
 94344,
 94346,
 100491,
 94347,
 100493,
 102542,
 94351,
 96395,
 94353,
 102546,
 94354,
 98453,
 102549,
 98455,
 102552,
 100505,
 102554,
 98459,
 94361,
 98461,
 96405,
 96409,
 94368,
 94369,
 94370,
 98467,
 100517,
 96423,
 102569,
 98477,
 98478,
 94385,
 98

While remaining datapoints have more or less exact calorie counts, we see that we have dropped some people from the original totals. This is because tots has rows for all people, including individuals who did not complete any dietary recall for day 1. So we are fine. 

Now, we are going to validate our aggregation against the aggregation done by the database originally. 

In [193]:
# AI-generated (Claude)

# Construct a translator between our column names (in sums) and USDA column names
tot_names = {7000: "FL_TOTAL", 7100: "FL_ANTHO", 7200: "FL_CATECHIN", 7300: "FL_3_OLS",
             7400: "FL_NONES", 7500: "FL_ONES", 7600: "FL_OLS", 7700: "FL_ISO"}
col_to_code = {f"FL_{t}": c for c, t in tags.items()}

def official_name(col, day):
    # food-group columns from fped
    if col in fped_cols:
        return f"DR{day}T_{col}"
    
    # flavonoid columns from flav
    code = col_to_code[col]
    return f"DR{day}T_{tot_names.get(code, f'FL{code}')}"

results = []
for day in (1, 2):
    # Sum all nutrients for each day
    sums = foods.loc[foods["RECALL_DAY"] == day].groupby("SEQN")[attr_cols].sum()

    # create a combined dataframe for FPED_TOT and FLAV_TOT (this is reference)
    official = pd.concat([
        data[f"FPED_TOT{day}"].set_index("SEQN"),
        data[f"FLAV_TOT{day}"].set_index("SEQN").drop(columns=f"DR{day}DRSTZ"),
    ], axis=1)

    # Map our columns to USDA columns, retain only these columns (not extra)
    names = {c: official_name(c, day) for c in attr_cols}
    official = official[list(names.values())]
    official.columns = list(names.keys())
    official = official.dropna(how="all")

    # Find common people, compute their aggregated differences
    common = sums.index.intersection(official.index)
    diff = (sums.loc[common] - official.loc[common]).abs()

    # Return summaries of aggregation
    # Locate values where we have large quantities of consumption
    big = official.loc[common] > 1
    rel = (diff / official.loc[common].abs()).where(big)

    # Display summaries
    results.append(pd.DataFrame({
        "day": day,
        "n_people": len(common),
        "max_abs_diff": diff.max(), # Maximum difference for this food
        "n_off_by_0.01": (diff > 0.01).sum(), # Number of people with consumption off for this food
        "mean_official": official.loc[common].mean(), # Average consumption
        "max_rel_diff": rel.max(), # Largest relative maximum difference
    }))

# Display table
report = pd.concat(results)

for d in (1, 2):
    have = set(foods.loc[foods["RECALL_DAY"] == d, "SEQN"])
    off = set(data[f"FPED_TOT{d}"].dropna(subset=[f"DR{d}T_F_TOTAL"])["SEQN"])
    print(f"Day {d}: reliable with foods={len(have)}, missing from official={len(have - off)}")

Day 1: reliable with foods=7483, missing from official=0
Day 2: reliable with foods=6502, missing from official=0


In [194]:
print(report.loc[report.index.isin(flav_cols)].sort_values("max_rel_diff", ascending=False).head(10).to_string())
print(report.loc[report.index.isin(fped_cols)].sort_values("max_abs_diff", ascending=False).head(10).to_string())

                  day  n_people  max_abs_diff  n_off_by_0.01  mean_official  max_rel_diff
FL_THEAFLAV33DG     2      6502      0.201600            356       0.860401      0.141053
FL_TOT_FLAVONE      1      7483      0.164168           1694       0.746877      0.137680
FL_EPICATEC         2      6502      0.157100            922       8.255897      0.108610
FL_MYRIC            2      6502      0.191600           1560       1.033241      0.094072
FL_THEAFLAV         2      6502      0.129200            306       0.786475      0.090350
FL_TOT_ISOFLAV      1      7483      0.126775            954       2.203003      0.088542
FL_TOT_FLAVONOL     1      7483      0.272400           3732      13.900358      0.079295
FL_TOT_ANTHOCYAN    1      7483      0.196338           1288      12.021121      0.070386
FL_KAEMF            1      7483      0.155100           2012       3.485002      0.068762
FL_KAEMF            2      6502      0.080000           1637       3.220677      0.067611
          

Looks like there is a lot of small rounding errors (the fact that rates were intially stored imprecisely) that slightly accumulated. It results in some large relative differences but minimal absolute differences (less than 0.4 mg).

Now, let us work to get our final dataset by aggregating:

In [195]:
# AI-generated (Claude)

# Retrieve all nutrient columns (which begin with KCAL)
nutrient_cols = list(iff1.columns[iff1.columns.get_loc("KCAL"):])

# Combine with attributes (such as FPED or flavanol)
diet_cols = nutrient_cols + attr_cols
assert foods[diet_cols].apply(pd.api.types.is_numeric_dtype).all()

# Group by person and day, obtaining their totals
day_totals = foods.groupby(["SEQN", "RECALL_DAY"])[diet_cols].sum().reset_index()

# Compute means for each column across the two days
diet = day_totals.groupby("SEQN")[diet_cols].mean()

# 
diet.insert(0, "N_DAYS", day_totals.groupby("SEQN").size())
diet.insert(1, "KCAL_D1", day_totals.loc[day_totals["RECALL_DAY"] == 1].set_index("SEQN")["KCAL"])

density_cols = [c for c in diet_cols if c != "KCAL"]
p1k = diet[density_cols].div(diet["KCAL"] / 1000, axis=0).add_suffix("_P1K")
diet = pd.concat([diet, p1k], axis=1).reset_index()

print(diet.shape)
print(diet["N_DAYS"].value_counts().to_dict())
print("Missing day 1:", diet["KCAL_D1"].isna().sum())
print(diet[["SEQN", "N_DAYS", "KCAL_D1", "KCAL", "F_TOTAL", "F_TOTAL_P1K"]].head())

(7495, 280)
{2: 6490, 1: 1005}
Missing day 1: 12
    SEQN  N_DAYS  KCAL_D1    KCAL   F_TOTAL  F_TOTAL_P1K
0  93704       2   1230.0  1293.0  1.467595     1.135031
1  93705       2   1202.0  1218.5  0.012600     0.010341
2  93706       1   1987.0  1987.0  0.000000     0.000000
3  93707       2   1775.0  1784.5  0.000000     0.000000
4  93708       2   1251.0  1046.5  0.243750     0.232919


/var/folders/wn/4vhmdq557_7f8325xdqxk2tr0000gn/T/ipykernel_99662/707595562.py:22: PerformanceWarning: DataFrame is highly fragmented.  This is usually the result of calling `frame.insert` many times, which has poor performance.  Consider joining all columns at once using pd.concat(axis=1) instead. To get a de-fragmented frame, use `newframe = frame.copy()`
  diet = pd.concat([diet, p1k], axis=1).reset_index()


In [196]:
# AI-generated (Claude)

# Retrieve all nutrient columns (which begin with KCAL)
nutrient_cols = list(iff1.columns[iff1.columns.get_loc("KCAL"):])

# Combine with attributes (such as FPED and flavanol)
diet_cols = nutrient_cols + attr_cols
assert foods[diet_cols].apply(pd.api.types.is_numeric_dtype).all() # Make sure no non numerics

# Group by day and person, aggregate
day_totals = foods.groupby(["SEQN", "RECALL_DAY"])[diet_cols].sum().reset_index()

# Obtain sex, map sex into our dataframe
sex = data["DEMO"].set_index("SEQN")["RIAGENDR"]
day_totals["SEX"] = day_totals["SEQN"].map(sex)

# Produce plausible bounds on consumption in a day
# Based on Willett's Nutritional Epidemiology
lo = day_totals["SEX"].map({1: 800, 2: 500})
hi = day_totals["SEX"].map({1: 4200, 2: 3500})
day_totals["PLAUSIBLE"] = day_totals["KCAL"].between(lo, hi)

# View all of the plausible diet counts for each recall day
print(day_totals.groupby("RECALL_DAY")["PLAUSIBLE"].value_counts().unstack())

# For each person, show how many plausible days exist
diet = day_totals.groupby("SEQN")[diet_cols].mean()
diet.insert(0, "N_DAYS", day_totals.groupby("SEQN").size())
diet.insert(2, "PLAUSIBLE_ALL", day_totals.groupby("SEQN")["PLAUSIBLE"].all())

# Locate column density, normalize and concatenate for rates
density_cols = [c for c in diet_cols if c != "KCAL"]
p1k = diet[density_cols].div(diet["KCAL"] / 1000, axis=0).add_suffix("_P1K")
diet = pd.concat([diet, p1k], axis=1).reset_index()


print(diet.shape)
print(diet["N_DAYS"].value_counts().to_dict())
print("Not plausible on all days:", (~diet["PLAUSIBLE_ALL"]).sum())
print(diet[["SEQN", "N_DAYS", "KCAL", "F_TOTAL", "F_TOTAL_P1K"]].head())

PLAUSIBLE   False  True 
RECALL_DAY              
1             538   6945
2             472   6030
(7495, 280)
{2: 6490, 1: 1005}
Not plausible on all days: 880
    SEQN  N_DAYS    KCAL   F_TOTAL  F_TOTAL_P1K
0  93704       2  1293.0  1.467595     1.135031
1  93705       2  1218.5  0.012600     0.010341
2  93706       1  1987.0  0.000000     0.000000
3  93707       2  1784.5  0.000000     0.000000
4  93708       2  1046.5  0.243750     0.232919


/var/folders/wn/4vhmdq557_7f8325xdqxk2tr0000gn/T/ipykernel_99662/3263270631.py:34: PerformanceWarning: DataFrame is highly fragmented.  This is usually the result of calling `frame.insert` many times, which has poor performance.  Consider joining all columns at once using pd.concat(axis=1) instead. To get a de-fragmented frame, use `newframe = frame.copy()`
  diet = pd.concat([diet, p1k], axis=1).reset_index()


# Constructing Demographics

Let's extract demographic details:

In [197]:
# Pick Person ID, Age, Sex, Race/Ethnicity, Education, Income-to-Poverty ratio (0-5), Pregnancy Status
demo = data["DEMO"][["SEQN", "RIDAGEYR", "RIAGENDR", "RIDRETH3", "DMDEDUC2", "INDFMPIR", "RIDEXPRG"]].copy()

# 7 and 9 are both missing codes
demo["DMDEDUC2"] = demo["DMDEDUC2"].replace([7, 9], np.nan)

Let's extract blood pressure details:

In [198]:
# Make a copy of data frame
bpx = data["BPX"].copy()

# Collect column headers
# systolic - diastolic
sys_cols = [f"BPXSY{i}" for i in range(1, 5)]
dia_cols = [f"BPXDI{i}" for i in range(1, 5)]

# 0 is a typical missing value
# not possible, means there was some sort of error 
bpx[dia_cols] = bpx[dia_cols].replace(0, np.nan)

# Take average of available readings for a person
bpx["SBP"] = bpx[sys_cols].mean(axis=1)
bpx["DBP"] = bpx[dia_cols].mean(axis=1)

# Keep only average and ID
bpx = bpx[["SEQN", "SBP", "DBP"]]

Let's extract smoking details:

In [199]:
# ID, smoked at last 100 ciagrettes in your life, are you now a smoker
smq = data["SMQ"][["SEQN", "SMQ020", "SMQ040"]].copy()

# Typical missing values
smq[["SMQ020", "SMQ040"]] = smq[["SMQ020", "SMQ040"]].replace([7, 9], np.nan)

# We will create a smoking category based on this previous data
smq["SMOKING"] = pd.NA

# SMOQ20 == 2 --> Less than 100 smokes in life 
# SMOQ50 only answered if SMOQ20 == 3
# SMOQ40 == 3 --> Don't smoke anymore
# SMOQ40 == 1 or 2 --> Smokes everyday, smokes occassionally
smq.loc[smq["SMQ020"] == 2, "SMOKING"] = "never"
smq.loc[smq["SMQ040"] == 3, "SMOKING"] = "former"
smq.loc[smq["SMQ040"].isin([1, 2]), "SMOKING"] = "current"

# Keep only engineered smoking column
smq = smq[["SEQN", "SMOKING"]]


Now, let's grab the physical activity:

In [200]:
# Vig_work --> columns corresponding to vigorous employment work minutes
# Mod_work --> columns corresponding to moderate employment work minutes
# Transport --> columns corresponding to transport minutes (bike or walking)
# Vig_Rec --> columns corresponding to vigorous recreation minutes
# Mod_Rec --> columns corresponding to moderate recreation minutes

# column, days per week, minutes per day

# MET --> Measures how hard an activity is compared with sitting at rest
# 1 MET --> Resting energy, 4 four times as much, 8 about 8 times
# Moderate activity usually 3-6 Mets, and virgorous activity is usually +6 Mets
# We will take an approximate
acts = {
    "VIG_WORK":  ("PAQ605", "PAQ610", "PAD615", 8),
    "MOD_WORK":  ("PAQ620", "PAQ625", "PAD630", 4),
    "TRANSPORT": ("PAQ635", "PAQ640", "PAD645", 4),
    "VIG_REC":   ("PAQ650", "PAQ655", "PAD660", 8),
    "MOD_REC":   ("PAQ665", "PAQ670", "PAD675", 4),
}


# Create an empty dataframe to fill
paq = data["PAQ"].copy()
met = pd.DataFrame(index=paq.index)
mins_per_day = pd.DataFrame(index=paq.index)


# Iterate over columns
for name, (yn, days, mins, weight) in acts.items():
    # Replace standard missigns
    ans = paq[yn].replace([7, 9], np.nan)
    d = paq[days].replace([77, 99], np.nan)
    m = paq[mins].replace([7777, 9999], np.nan)

    # Compute total met
    # .where() ensures that individuals who answered no are not marked as NA but 0
    # Resulting columns either are evaluated, have 0 if person responded no, or have none if no answer
    met[name] = (d * m * weight).where(ans == 1, 0).where(ans.notna())

    # Add counts to mins per day df
    mins_per_day[name] = m.where(ans == 1, 0)

# More than 16 hours a day....
# Filter them out
too_much = mins_per_day.sum(axis=1) > 960
print("Over 16 hours/day:", too_much.sum())
met = met.where(~too_much)

# Aggregate total MET minutes
paq["MET_MIN_WK"] = met.sum(axis=1, min_count=len(acts))

# Discretize (for later use)
paq["MEETS_PA"] = (paq["MET_MIN_WK"] >= 600).astype(float).where(paq["MET_MIN_WK"].notna())

# How many sedentary minutes
paq["SED_MIN"] = paq["PAD680"].replace([7777, 9999], np.nan)

# Sub-select as desired
paq = paq[["SEQN", "MET_MIN_WK", "MEETS_PA", "SED_MIN"]]

print(paq.describe().to_string())

Over 16 hours/day: 79
                SEQN    MET_MIN_WK     MEETS_PA      SED_MIN
count    5856.000000   5725.000000  5725.000000  5811.000000
mean    98280.043204   3855.658341     0.609432   331.679917
std      2683.291503   6157.596519     0.487920   199.148637
min     93705.000000      0.000000     0.000000     0.000000
25%     95954.750000      0.000000     0.000000   180.000000
50%     98263.000000   1200.000000     1.000000   300.000000
75%    100594.750000   4700.000000     1.000000   480.000000
max    102956.000000  41440.000000     1.000000  1320.000000


Diabetes next:

In [201]:
# Diabetes diagnosis, insulin medication, diabetic pills
diq = data["DIQ"][["SEQN", "DIQ010", "DIQ050", "DIQ070"]].copy()

# Standard missing
diq[["DIQ010", "DIQ050", "DIQ070"]] = diq[["DIQ010", "DIQ050", "DIQ070"]].replace([7, 9], np.nan)

Now, here comes the big join!

In [202]:
# All of our parts, including some directly from the raw data
# LBXGH --> HBA1c (%)
# LBDHDD --> HDL cholesterol (mg/dL)
# LBXTC --> Total Cholesterol (mg/dL)
# LBXHSCRP --> C-reactive protein (mg/L)
# BMXBMI --> BMI 
# BMXWAIST --> Waist Circumference (cm)
parts = [
    data["GHB"][["SEQN", "LBXGH"]],
    data["HDL"][["SEQN", "LBDHDD"]],
    data["TCHOL"][["SEQN", "LBXTC"]],
    data["HSCRP"][["SEQN", "LBXHSCRP"]],
    data["BMX"][["SEQN", "BMXBMI", "BMXWAIST"]],
    bpx, smq, paq,
    data["FSQ"][["SEQN", "FSDHH"]],
    diq,
]

In [203]:
person = demo

# Succesively merge using SEQN
for p in parts:
    person = person.merge(p, on="SEQN", how="left", validate="1:1")

In [204]:
# Make pregnant flag, diabetes flag, and diabetes medication flag
person["PREGNANT"] = person["RIDEXPRG"] == 1
person["DIABETES_DX"] = person["DIQ010"] == 1
person["DIABETES_MED"] = (person["DIQ050"] == 1) | (person["DIQ070"] == 1)

In [205]:
print(person.shape)
print(person.isna().sum().to_string())

(9254, 26)
SEQN               0
RIDAGEYR           0
RIAGENDR           0
RIDRETH3           0
DMDEDUC2        3698
INDFMPIR        1231
RIDEXPRG        8144
LBXGH           3209
LBDHDD          2516
LBXTC           2516
LBXHSCRP        2004
BMXBMI          1249
BMXWAIST        1653
SBP             2537
DBP             2618
SMOKING         3398
MET_MIN_WK      3529
MEETS_PA        3529
SED_MIN         3443
FSDHH            492
DIQ010           361
DIQ050           358
DIQ070          7582
PREGNANT           0
DIABETES_DX        0
DIABETES_MED       0


Nothing here appears strange. Many of the areas with a large number of missings (such as DIQ070) are conditional and therefore are missing if not applicable. Similarly, education, pregnancy, physical activity, and smoking are asked to a smaller demographic (namely adults or subsets of adults), not everyone. Income has 12% missing, which is important for later.

Lets merge the final product now!

In [206]:
# Perform a straight merge
df = person.merge(diet, on="SEQN", how="left", validate="1:1")

# Exclusion steps
steps = [
    ("Adults 20+",                    df["RIDAGEYR"] >= 20),
    ("Has HbA1c",                     df["LBXGH"].notna()),
    ("Reliable recall",               df["SEQN"].isin(diet["SEQN"])),
    ("Not pregnant",                  ~df["PREGNANT"]),
    ("No diagnosed diabetes",         ~df["DIABETES_DX"]),
    ("No diabetes medication",        ~df["DIABETES_MED"]),
    ("Plausible calories, all days",  df["PLAUSIBLE_ALL"] == True),
]

# Document exclusion process
keep = pd.Series(True, index=df.index)
rows = []

# Fix a condition
for label, cond in steps:
    # Before the exclusion
    before = keep.sum()

    # Perform exclusion
    keep &= cond

    # Compute exclusion, loss, and remaining
    rows.append((label, before - keep.sum(), keep.sum()))

# Construct cascade dataframe
cascade = pd.DataFrame(rows, columns=["step", "removed", "remaining"])

# Remaining people
analytic = df[keep].reset_index(drop=True)


In [207]:
controls = ["DMDEDUC2", "INDFMPIR", "SMOKING", "MET_MIN_WK", "SED_MIN", "FSDHH"]
print(analytic[controls].isna().sum().to_string())
print("Complete on all controls:", analytic[controls].notna().all(axis=1).sum())

DMDEDUC2        5
INDFMPIR      388
SMOKING         0
MET_MIN_WK     68
SED_MIN        17
FSDHH         138
Complete on all controls: 2831


In [208]:
print(cascade)

                           step  removed  remaining
0                    Adults 20+     3685       5569
1                     Has HbA1c      551       5018
2               Reliable recall      457       4561
3                  Not pregnant       43       4518
4         No diagnosed diabetes      728       3790
5        No diabetes medication       60       3730
6  Plausible calories, all days      444       3286


In [209]:
analytic.shape

(3286, 305)

Now, we save!

In [210]:
# Output directories
raw_out = Path("data/raw_csv/2017_2018")
proc_out = Path("data/processed")
raw_out.mkdir(parents=True, exist_ok=True)
proc_out.mkdir(parents=True, exist_ok=True)

# Raw data files as CSVs
for name, df_ in data.items():
    df_.to_csv(raw_out / f"{name}.csv", index=False)

# All the merged tables, including analytic_cohort (main data)
tables = {
    "foods_long.csv.gz": foods,
    "diet_person.csv": diet,
    "person_details.csv": person,
    "analytic_cohort.csv": analytic,
    "exclusion_cascade.csv": cascade,
}
for fname, df_ in tables.items():
    df_.to_csv(proc_out / fname, index=False)

old = proc_out / "foods_long.csv"
if old.exists():
    old.unlink()

# Display sizes
for folder in (raw_out, proc_out):
    print(f"\n{folder}")
    for f in sorted(folder.glob("*.csv")):
        print(f"  {f.name:28} {f.stat().st_size / 1e6:8.1f} MB")


data/raw_csv/2017_2018
  BMX.csv                           0.5 MB
  BPX.csv                           0.6 MB
  DEMO.csv                          1.8 MB
  DIQ.csv                           0.7 MB
  DR1IFF.csv                       46.7 MB
  DR1TOT.csv                        6.6 MB
  DR2IFF.csv                       38.9 MB
  DRXFCD.csv                        0.7 MB
  FLAV_DESC.csv                     0.0 MB
  FLAV_FOODS.csv                    0.5 MB
  FLAV_TOT1.csv                     1.8 MB
  FLAV_TOT2.csv                     1.6 MB
  FLAV_VAL.csv                     11.4 MB
  FPED_DB.csv                       1.5 MB
  FPED_TOT1.csv                     2.3 MB
  FPED_TOT2.csv                     2.1 MB
  FSQ.csv                           1.0 MB
  GHB.csv                           0.1 MB
  HDL.csv                           0.1 MB
  HSCRP.csv                         0.1 MB
  PAQ.csv                           0.3 MB
  SMQ.csv                           0.5 MB
  TCHOL.csv                   

Just have to check the compressed file is also there:

In [211]:
f = proc_out / "foods_long.csv.gz"
print(f.exists(), f"{f.stat().st_size / 1e6:.1f} MB")

True 24.3 MB


# Food Table

In [212]:
list(raw.glob("*FNDDS*"))

[PosixPath('data/raw/2017_2018/2017-2018 FNDDS At A Glance - Ingredient Nutrient Values.xlsx'),
 PosixPath('data/raw/2017_2018/2017-2018 FNDDS At A Glance - FNDDS Ingredients.xlsx'),
 PosixPath('data/raw/2017_2018/2017-2018 FNDDS At A Glance - FNDDS Nutrient Values.xlsx')]

In [213]:
# File names
fndds_files = {
    "FNDDS_ING": "2017-2018 FNDDS At A Glance - FNDDS Ingredients.xlsx",
    "FNDDS_NUT": "2017-2018 FNDDS At A Glance - FNDDS Nutrient Values.xlsx",
}

# Read files in, display
for name, file in fndds_files.items():
    df_ = pd.read_excel(raw / file, header=1)
    df_.columns = df_.columns.str.upper()
    data[name] = df_
    print(f"\n=== {name} {df_.shape} ===")
    print(df_.columns.tolist())
    print(df_.head(3).to_string())


=== FNDDS_ING (24896, 10) ===
['FOOD CODE', 'MAIN FOOD DESCRIPTION', 'WWEIA CATEGORY NUMBER', 'WWEIA CATEGORY DESCRIPTION', 'SEQ NUM', 'INGREDIENT CODE', 'INGREDIENT DESCRIPTION', 'INGREDIENT WEIGHT (G)', 'RETENTION CODE', 'MOISTURE CHANGE (%)']
   FOOD CODE MAIN FOOD DESCRIPTION  WWEIA CATEGORY NUMBER WWEIA CATEGORY DESCRIPTION  SEQ NUM  INGREDIENT CODE                                                    INGREDIENT DESCRIPTION  INGREDIENT WEIGHT (G)  RETENTION CODE  MOISTURE CHANGE (%)
0   11000000           Milk, human                   9602                 Human milk        1             1107                                                Milk, human, mature, fluid                  100.0               0                  0.0
1   11100000             Milk, NFS                   1004          Milk, reduced fat        1             1077                          Milk, whole, 3.25% milkfat, with added vitamin D                   35.0               0                  0.0
2   11100000      

In [214]:
data["FNDDS_NUT"].columns

Index(['FOOD CODE', 'MAIN FOOD DESCRIPTION', 'WWEIA CATEGORY NUMBER',
       'WWEIA CATEGORY DESCRIPTION', 'ENERGY (KCAL)', 'PROTEIN (G)',
       'CARBOHYDRATE (G)', 'SUGARS, TOTAL\n(G)', 'FIBER, TOTAL DIETARY (G)',
       'TOTAL FAT (G)', 'FATTY ACIDS, TOTAL SATURATED (G)',
       'FATTY ACIDS, TOTAL MONOUNSATURATED (G)',
       'FATTY ACIDS, TOTAL POLYUNSATURATED (G)', 'CHOLESTEROL (MG)',
       'RETINOL (MCG)', 'VITAMIN A, RAE (MCG_RAE)', 'CAROTENE, ALPHA (MCG)',
       'CAROTENE, BETA (MCG)', 'CRYPTOXANTHIN, BETA (MCG)', 'LYCOPENE (MCG)',
       'LUTEIN + ZEAXANTHIN (MCG)', 'THIAMIN (MG)', 'RIBOFLAVIN (MG)',
       'NIACIN (MG)', 'VITAMIN B-6 (MG)', 'FOLIC ACID (MCG)',
       'FOLATE, FOOD (MCG)', 'FOLATE, DFE (MCG_DFE)', 'FOLATE, TOTAL (MCG)',
       'CHOLINE, TOTAL (MG)', 'VITAMIN B-12 (MCG)',
       'VITAMIN B-12, ADDED\n(MCG)', 'VITAMIN C (MG)',
       'VITAMIN D (D2 + D3) (MCG)', 'VITAMIN E (ALPHA-TOCOPHEROL) (MG)',
       'VITAMIN E, ADDED\n(MG)', 'VITAMIN K (PHYLLOQUINONE) (

Based on the output above, we see that the columns not only do not have the same name, but these columns are also out of order. I used Claude to construct a mapping between the columns:

In [215]:
fndds_to_short = {
    "ENERGY (KCAL)": "KCAL", "PROTEIN (G)": "PROT", "CARBOHYDRATE (G)": "CARB",
    "SUGARS, TOTAL (G)": "SUGR", "FIBER, TOTAL DIETARY (G)": "FIBE", "TOTAL FAT (G)": "TFAT",
    "FATTY ACIDS, TOTAL SATURATED (G)": "SFAT", "FATTY ACIDS, TOTAL MONOUNSATURATED (G)": "MFAT",
    "FATTY ACIDS, TOTAL POLYUNSATURATED (G)": "PFAT", "CHOLESTEROL (MG)": "CHOL",
    "RETINOL (MCG)": "RET", "VITAMIN A, RAE (MCG_RAE)": "VARA", "CAROTENE, ALPHA (MCG)": "ACAR",
    "CAROTENE, BETA (MCG)": "BCAR", "CRYPTOXANTHIN, BETA (MCG)": "CRYP", "LYCOPENE (MCG)": "LYCO",
    "LUTEIN + ZEAXANTHIN (MCG)": "LZ", "THIAMIN (MG)": "VB1", "RIBOFLAVIN (MG)": "VB2",
    "NIACIN (MG)": "NIAC", "VITAMIN B-6 (MG)": "VB6", "FOLIC ACID (MCG)": "FA",
    "FOLATE, FOOD (MCG)": "FF", "FOLATE, DFE (MCG_DFE)": "FDFE", "FOLATE, TOTAL (MCG)": "FOLA",
    "CHOLINE, TOTAL (MG)": "CHL", "VITAMIN B-12 (MCG)": "VB12", "VITAMIN B-12, ADDED (MCG)": "B12A",
    "VITAMIN C (MG)": "VC", "VITAMIN D (D2 + D3) (MCG)": "VD",
    "VITAMIN E (ALPHA-TOCOPHEROL) (MG)": "ATOC", "VITAMIN E, ADDED (MG)": "ATOA",
    "VITAMIN K (PHYLLOQUINONE) (MCG)": "VK", "CALCIUM (MG)": "CALC", "PHOSPHORUS (MG)": "PHOS",
    "MAGNESIUM (MG)": "MAGN", "IRON (MG)": "IRON", "ZINC (MG)": "ZINC", "COPPER (MG)": "COPP",
    "SELENIUM (MCG)": "SELE", "POTASSIUM (MG)": "POTA", "SODIUM (MG)": "SODI",
    "CAFFEINE (MG)": "CAFF", "THEOBROMINE (MG)": "THEO", "ALCOHOL (G)": "ALCO",
    "4:0 (G)": "S040", "6:0 (G)": "S060", "8:0 (G)": "S080", "10:0 (G)": "S100",
    "12:0 (G)": "S120", "14:0 (G)": "S140", "16:0 (G)": "S160", "18:0 (G)": "S180",
    "16:1 (G)": "M161", "18:1 (G)": "M181", "20:1 (G)": "M201", "22:1 (G)": "M221",
    "18:2 (G)": "P182", "18:3 (G)": "P183", "18:4 (G)": "P184", "20:4 (G)": "P204",
    "20:5 N-3 (G)": "P205", "22:5 N-3 (G)": "P225", "22:6 N-3 (G)": "P226", "WATER (G)": "MOIS",
}

In [216]:
# Clean awkward name line breaks
nut = data["FNDDS_NUT"].copy()
nut.columns = [" ".join(c.split()) for c in nut.columns]

# Rename columns using the mapping (plus some of the meta names)
nut = nut.rename(columns={
    "FOOD CODE": "FOODCODE", "MAIN FOOD DESCRIPTION": "FOOD_DESC",
    "WWEIA CATEGORY NUMBER": "WWEIA_CODE", "WWEIA CATEGORY DESCRIPTION": "WWEIA_DESC",
    **fndds_to_short,
})

# Coerce foodcode
nut["FOODCODE"] = nut["FOODCODE"].astype(int)

# Make sure we have 65 nutrients
assert len(fndds_to_short) == 65

# Make sure nutrient_cols (from the analytic cohort) are identical 
assert set(fndds_to_short.values()) == set(nutrient_cols)

# Make sure a food only appears once
assert nut["FOODCODE"].is_unique

# Keep the meta columns + nutrition labels
# Merge with fped categories and flavanols
food_table = (nut[["FOODCODE", "FOOD_DESC", "WWEIA_CODE", "WWEIA_DESC"] + nutrient_cols]
              .merge(fped, on="FOODCODE", how="left", validate="1:1")
              .merge(flav, on="FOODCODE", how="left", validate="1:1"))

# Every food has a value for all 139 attributes
assert food_table[nutrient_cols + attr_cols].notna().all().all()

# Dispaly shape and head
print(food_table.shape)
print(food_table[["FOODCODE", "FOOD_DESC", "WWEIA_DESC", "KCAL", "F_TOTAL", "FL_TOT_FLAVONOIDS"]].head())

(7083, 143)
   FOODCODE                       FOOD_DESC         WWEIA_DESC  KCAL  F_TOTAL  \
0  11000000                     Milk, human         Human milk    70      0.0   
1  11100000                       Milk, NFS  Milk, reduced fat    51      0.0   
2  11111000                     Milk, whole        Milk, whole    60      0.0   
3  11111100         Milk, low sodium, whole        Milk, whole    61      0.0   
4  11111150  Milk, calcium fortified, whole        Milk, whole    60      0.0   

   FL_TOT_FLAVONOIDS  
0                0.0  
1                0.0  
2                0.0  
3                0.0  
4                0.0  


Now, we should validate that the nutritional scales presented in this file match the scales in the diet recall:

In [217]:
# AI-generated (Claude)

# Join food-people with foods
chk = foods[["FOODCODE", "GRAMS"] + nutrient_cols].merge(
    food_table[["FOODCODE"] + nutrient_cols], on="FOODCODE", suffixes=("", "_REF"), validate="m:1")

# Scale each nutrient by the grams eaten
ref = chk[[c + "_REF" for c in nutrient_cols]].mul(chk["GRAMS"] / 100, axis=0)
ref.columns = nutrient_cols

# Compute difference and relative scale
diff = (chk[nutrient_cols] - ref).abs()
scale = chk[nutrient_cols].mean()

# Display differences
report = pd.DataFrame({"max_abs_diff": diff.max(), "mean_value": scale,
                       "max_diff_pct_of_mean": diff.max() / scale * 100})
print(report.sort_values("max_diff_pct_of_mean", ascending=False).head(10).to_string())

      max_abs_diff  mean_value  max_diff_pct_of_mean
CAFF      27.64000    6.546850            422.187798
THEO       9.04000    2.353037            384.184361
VD         1.10000    0.317956            345.960047
ALCO       0.77000    0.331173            232.506641
FF        22.60000   12.426037            181.876163
FIBE       1.86800    1.046536            178.493580
CRYP       7.64000    5.924365            128.958974
FOLA      22.60000   24.355958             92.790438
ATOA       0.05425    0.059805             90.712073
P184       0.00050    0.000556             89.848737


These percentage differences are a bit alarming. Let's investigate further:

In [218]:
# AI-generated (Claude)

# Extract vals and relative differences
vals = chk[nutrient_cols]
rel = diff / vals.where(vals > 0)

# Compute a dataframe containing all off by atleast one percent + total off by at least one percent + percentage off by one percent
summary = pd.DataFrame({
    "n_foods_with_value": (vals > 0).sum(),
    "n_off_by_1pct": (rel > 0.01).sum(),
    "share_off_by_1pct": (rel > 0.01).sum() / (vals > 0).sum(),
})
print(summary.sort_values("share_off_by_1pct", ascending=False).head(10).to_string())

# Find the worst points for each of CAFF, VD, and FIBE
for n in ["CAFF", "VD", "FIBE"]:
    worst = diff[n].nlargest(5).index
    out = chk.loc[worst, ["FOODCODE", "GRAMS", n]].assign(TABLE_SCALED=ref.loc[worst, n])
    out = out.merge(food_table[["FOODCODE", "FOOD_DESC"]], on="FOODCODE")
    print(f"\n=== {n} ===")
    print(out.to_string(index=False))

      n_foods_with_value  n_off_by_1pct  share_off_by_1pct
P205               33534          29647           0.884088
P184               11192           9847           0.879825
P225               39843          33790           0.848079
M221               31484          25839           0.820703
VD                 52263          41404           0.792224
CRYP               37392          28161           0.753129
P226               27710          19961           0.720354
FF                133955          96439           0.719936
FIBE              107456          76024           0.707490
MAGN              182829         128299           0.701743

=== CAFF ===
 FOODCODE  GRAMS   CAFF  TABLE_SCALED                                           FOOD_DESC
 92308000 5952.0 1099.0       1071.36  Tea, iced, brewed, black, pre-sweetened with sugar
 92308000 3968.0  733.0        714.24  Tea, iced, brewed, black, pre-sweetened with sugar
 92308000 3968.0  733.0        714.24  Tea, iced, brewed, black, pr

In [219]:
# AI-generated (Claude)

# Merge food table with people-food, scale by reference by consumed quantity
c2 = foods[["SEQN", "RECALL_DAY", "FOODCODE", "GRAMS"] + nutrient_cols].merge(
    food_table[["FOODCODE"] + nutrient_cols], on="FOODCODE", suffixes=("", "_REF"), validate="m:1")
ref_cols = [n + "_REF" for n in nutrient_cols]
c2[ref_cols] = c2[ref_cols].mul(c2["GRAMS"] / 100, axis=0)

# Groupby and sum people's nutrition
day = c2.groupby(["SEQN", "RECALL_DAY"])[nutrient_cols + ref_cols].sum()

# Compute all discrepancies for non-zero nutrients for people days
# AKA --> For each-person day with non-zero intake, compute the relative discrepancy
rel_day = pd.DataFrame({n: (day[n] - day[n + "_REF"]).abs() / day[n].where(day[n] > 0)
                        for n in nutrient_cols})

# Display summary of distribution for top discrepencies
print(pd.DataFrame({"median_pct": rel_day.median() * 100,
                    "p95_pct": rel_day.quantile(0.95) * 100})
      .sort_values("p95_pct", ascending=False).head(10).round(2).to_string())

      median_pct  p95_pct
P184        8.00    40.00
CRYP        1.55    30.00
M221        2.63    28.30
P226        1.69    26.87
ACAR        0.88    25.00
P205        4.00    24.84
VD          1.90    23.57
CAFF        0.62    22.50
P225        2.12    13.58
THEO        0.61    10.00


Using these published reference values seem maybe a little unreliable based on these scans; the problem is that the reference values are rounded but were not rounded in the original calculation that takes place in the food-person rows. As such, we will try a recovery strategy by obtaining all of the estimates from the food-people rows to see if we can preserve some precision that may have been lost. The key observation is that NHANES multiplied each food by some reference and then rounded the resulting value to the nearest unit. To recover per-100 g values, each stored value is multiplied by 100 / grams. The IFF rounds every stored value by the same fixed amount, so this factor also scales the rounding error. For small servings (under 100 g) the factor is greater than 1 and amplifies the error; for large servings it is less than 1 and shrinks it. We therefore recover each food's values from its largest recorded serving.

In [220]:
# AI-generated (Claude)

# Keep records with positive weight
pos = foods[foods["GRAMS"] > 0]

# Find largest serving (make sure over 100 or just stick with default)
largest = pos.loc[pos.groupby("FOODCODE")["GRAMS"].idxmax(), ["FOODCODE", "GRAMS"] + nutrient_cols]
largest = largest[largest["GRAMS"] > 100]

# Recover per 100 g serving
recovered = largest[nutrient_cols].div(largest["GRAMS"] / 100, axis=0)
recovered.index = largest["FOODCODE"].values

# Find all over 100 for a given food, recover their per 100 g serving
big = pos[pos["GRAMS"] > 100]
per100 = big[nutrient_cols].div(big["GRAMS"] / 100, axis=0)
per100["FOODCODE"] = big["FOODCODE"].values

# Compute relative spread of these recovered values 
g = per100.groupby("FOODCODE")[nutrient_cols]
rel_spread = (g.max() - g.min()) / g.median().where(g.median() > 0)

# Output results
print("Foods eaten:", pos["FOODCODE"].nunique())
print("Foods recovered (largest portion > 100 g):", len(recovered))
print(rel_spread.quantile([0.5, 0.99]).T.sort_values(0.99, ascending=False).head(10).round(4).to_string())

# Write recovered values into table
# If values are not recovered, keep default
ft = food_table.set_index("FOODCODE")
ft[nutrient_cols] = ft[nutrient_cols].astype(float)
ft.update(recovered)

# Note source of our recovery
ft.insert(3, "NUTR_SOURCE", np.where(ft.index.isin(recovered.index), "NHANES_IFF", "FNDDS_ROUNDED"))
food_table = ft.reset_index()

# Make sure no columns are NA, display value counts
assert food_table[nutrient_cols].notna().all().all()
print(food_table["NUTR_SOURCE"].value_counts().to_string())

Foods eaten: 5126
Foods recovered (largest portion > 100 g): 3892
        0.50    0.99
CRYP  0.0255  2.0000
VD    0.0400  2.0000
LYCO  0.0002  1.3852
VB12  0.0037  1.3125
VC    0.0034  1.0327
CAFF  0.0166  1.0015
RET   0.0051  0.8910
LZ    0.0017  0.8718
ACAR  0.0043  0.8091
P184  0.0491  0.6580
NUTR_SOURCE
NHANES_IFF       3892
FNDDS_ROUNDED    3191


The best way to assess our recovery is by comparing it to the imprecise baseline:

In [226]:
# AI-generated (Claude)
# Retrieve public and recovered references
pub = nut.set_index("FOODCODE").loc[recovered.index, nutrient_cols].astype(float)
grams = largest.set_index("FOODCODE").loc[recovered.index, "GRAMS"]

# Determines number of decimals used by USDA for a particular nutrient
def decimals(col):
    for d in range(5):
        if np.allclose(col * 10**d, np.round(col * 10**d)):
            return d
    return 5

# Figure out decimals for each nutrient, identify the smallest possible step in that precision
dec = pd.Series({n: decimals(pub[n]) for n in nutrient_cols})
unit = 10.0 ** -dec

# Tol --> Largest gap explained by rounding precision
# USDA rounding (0.5 +- unit) and rounding in recovery (scaled down by our multiplication factor 100/grams)
tol = pd.DataFrame({n: 0.5 * unit[n] * (1 + 100 / grams) + 1e-9 for n in nutrient_cols})

# Perform difference
gap = (recovered[nutrient_cols] - pub).abs()

# Locate food-nutrients that exceed rounding tolerance
bad = gap > tol

# Summarize results
print(pd.DataFrame({"decimals": dec, "n_mismatch": bad.sum(), "max_gap": gap.max()})
      .sort_values("n_mismatch", ascending=False).head(10).to_string())
print("\nFoods with any mismatch:", bad.any(axis=1).sum(), "of", len(bad))

# Pull specific examples to examine
examples = [11111000, 92101000, 63101000]
for fc in [f for f in examples if f in recovered.index]:
    print(f"\n{fc}  {nut.set_index('FOODCODE').loc[fc, 'FOOD_DESC']}")
    print(pd.DataFrame({"published": pub.loc[fc, ["KCAL", "PROT", "VC", "VD", "CAFF", "POTA"]],
                        "recovered": recovered.loc[fc, ["KCAL", "PROT", "VC", "VD", "CAFF", "POTA"]]}).round(4).to_string())

      decimals  n_mismatch   max_gap
MOIS         2          14  0.008972
ZINC         2          12  0.009643
TFAT         2           9  0.008962
IRON         2           7  0.010102
PROT         2           6  0.009339
SUGR         2           5  0.009996
ATOC         2           4  0.009195
CARB         2           4  0.009524
VB12         2           2  0.009231
LZ           0           1  0.920899

Foods with any mismatch: 63 of 3892

11111000  Milk, whole
      published  recovered
KCAL      60.00    59.9898
PROT       3.28     3.2802
VC         0.00     0.0000
VD         1.10     1.1014
CAFF       0.00     0.0000
POTA     150.00   150.0000

92101000  Coffee, brewed
      published  recovered
KCAL       1.00       1.00
PROT       0.12       0.12
VC         0.00       0.00
VD         0.00       0.00
CAFF      40.00      40.00
POTA      49.00      49.00

63101000  Apple, raw
      published  recovered
KCAL      52.00      52.00
PROT       0.26       0.26
VC         4.60       4.60

Mismatches appear very small. Even the "bad" instances are right on the boundary of tolerated precision, suggesting some floating point imprecision driving it over the boundary. We should be ready to save.

In [227]:
food_table.to_csv(proc_out / "food_table.csv", index=False)